# Hierarchy Recovery — Counterfactual Necessity: FULL RUN

Scaled up from the 15-event pilot (which passed: AUC=0.750 after leak-filtering) to the full eligible pool (~140ish events, check the printed 'Eligible pool' count in Step 1 for the exact number -- N_PILOT=500 is just set higher than the real pool so everything eligible gets taken).

**New in this version**:
- Step 1 takes the full eligible pool instead of a 15-event stratified sample.
- Step 2 now has **resume support** -- safe to interrupt and re-run; it skips events already completed and skips loading a model entirely if all its events are already done. Important at this scale (4 model loads, ~140 events) since a Colab timeout would otherwise silently redo everything.
- Step 3 still filters leaked rollouts before computing necessity (same as before).

**Requires**: `kill_screen_events.json`, `hidden_reference_labels.csv`, and `hidden_reference_labels_round2.csv` already in `MyDrive/llm_planning_study/hand_labeling_export/`.

## Setup

In [ ]:
!pip install transformers accelerate scikit-learn scipy -q

In [ ]:
from google.colab import drive
drive.mount("/content/gdrive")

In [ ]:
# ---- scope_flag (inlined; no file to place) ---------------------------------
# Installs a module named `scope_flag` so the gated cells below resolve their
# `import scope_flag` without any file on disk.
#
# SET THIS ONE TOGGLE:
SCOPE_ANALYSIS_ENABLED = False   # False = paper path (default)
#
# Governs scope ANALYSIS AND REPORTING only, never sample construction. Step 1
# below draws its events stratified on the human local/global labels, and no
# flag can change that. See SCOPE_ISOLATION.md section 3.

import sys, types

_sf = types.ModuleType("scope_flag")
_sf.SCOPE_ANALYSIS_ENABLED = SCOPE_ANALYSIS_ENABLED
_sf.SCOPE_BANNER_SHOWN = False

def _scope_banner():
    if _sf.SCOPE_BANNER_SHOWN:
        return _sf.SCOPE_ANALYSIS_ENABLED
    _sf.SCOPE_BANNER_SHOWN = True
    if _sf.SCOPE_ANALYSIS_ENABLED:
        print("\n[scope_flag] SCOPE_ANALYSIS_ENABLED=True — retired scope "
              "sections WILL run. Not reported in the paper.")
    else:
        print("\n[scope_flag] SCOPE_ANALYSIS_ENABLED=False — scope sections "
              "skipped (paper path).")
    return _sf.SCOPE_ANALYSIS_ENABLED

def _skip_note(name):
    print(f"\n[scope_flag] SKIPPED (retired scope analysis): {name}")
    print("[scope_flag]   set SCOPE_ANALYSIS_ENABLED = True above to run it")

_sf.scope_banner = _scope_banner
_sf.skip_note = _skip_note
sys.modules["scope_flag"] = _sf
print(f"scope_flag installed. SCOPE_ANALYSIS_ENABLED = {SCOPE_ANALYSIS_ENABLED}")

## Step 1 — Select events (full eligible pool)

> **On the event draw below.** It is stratified to balance the human local/global annotation categories, so the resulting sample is **not** a random sample of confirmed revision events. This is disclosed rather than switched off: changing it would change which events every downstream result was computed on. It is harmless for the validity controls, which match each revision to a filler at the same marker word in the same model — but the paper must not describe this sample as random. See `SCOPE_ISOLATION.md` §3.


In [ ]:
# =============================================================================
# PILOT EVENT SELECTION -- counterfactual necessity pilot
#
# Re-derives real (model, problem_id, char_start, char_end) for a stratified
# subset of the 142 confidently-human-labeled events used in the embedding
# kill screen. The kill-screen JSON only kept derived sentence text, not these
# identifiers -- this script recovers them by matching back against the real
# labels/*.json and traces/*.json on Drive, the same way round-1's exclusion
# list and round-2's context extraction did.
#
# Filters to events where the ORIGINAL (with-revision) trace reached the
# correct answer (design decision #3) -- necessity is only meaningful there.
#
# Output: a JSON with, per pilot event: model, problem_id, char_start,
# marker_phrase, human_label, prefix_text (prompt + full_response up to
# char_start, ready to feed a suppressed-generation loop), and answer_gt.
# Does NOT run any generation -- that step is blocked on your inference setup.
# =============================================================================

from pathlib import Path
import json, random
from collections import defaultdict

GDRIVE_BASE = Path("/content/gdrive/MyDrive/llm_planning_study")
LABELS_DIR = GDRIVE_BASE / "labels"
TRACES_DIR = GDRIVE_BASE / "traces"
EXPORT_DIR = GDRIVE_BASE / "hand_labeling_export"

KILL_SCREEN_EVENTS_PATH = EXPORT_DIR / "kill_screen_events.json"
HIDDEN_R1_PATH = EXPORT_DIR / "hidden_reference_labels.csv"
HIDDEN_R2_PATH = EXPORT_DIR / "hidden_reference_labels_round2.csv"

N_PILOT = 500  # scaled up from the 15-event pilot -- set deliberately higher than
                # the actual eligible pool size (~140ish after all filters) so the
                # existing balancing logic takes EVERYTHING available in each class,
                # rather than subsampling. Confirm the real pool size in the printed
                # "Eligible pool" line once this runs -- it's not literally 500 events.
SEED = 2024

assert KILL_SCREEN_EVENTS_PATH.exists(), f"{KILL_SCREEN_EVENTS_PATH} not found"

with open(KILL_SCREEN_EVENTS_PATH) as f:
    kill_events = json.load(f)
print(f"Loaded {len(kill_events)} candidate events from the kill screen set.")

# ---- Need model + marker_phrase per (round, eid) to search labels/ with.
# kill_screen_events.json doesn't carry marker_phrase (it was consumed during
# sentence-splitting), so pull it back from the hidden-reference CSVs, which
# do have it, keyed the same way (round, event_id).
import csv

def load_hidden_reference(path, round_num):
    rows = {}
    if not path.exists():
        return rows
    with open(path) as f:
        for r in csv.DictReader(f):
            eid = int(r["event_id"])
            rows[(round_num, eid)] = r
    return rows

hidden = {}
hidden.update(load_hidden_reference(HIDDEN_R1_PATH, 1))
hidden.update(load_hidden_reference(HIDDEN_R2_PATH, 2))
print(f"Hidden reference rows available: {len(hidden)}")

# ---- Index labels/*.json by (model, problem_id) for fast lookup -----------
print("Indexing labels/ ...")
labels_index = {}   # (model, problem_id) -> full label object
for model_dir in LABELS_DIR.iterdir():
    if not model_dir.is_dir():
        continue
    for f in model_dir.glob("*.json"):
        with open(f) as fh:
            obj = json.load(fh)
        pid = obj.get("problem_id", f.stem)
        labels_index[(model_dir.name, pid)] = obj

trace_index = {}
for model_dir in TRACES_DIR.iterdir():
    if not model_dir.is_dir():
        continue
    for f in model_dir.glob("*.json"):
        trace_index[(model_dir.name, f.stem)] = f

# ---- For each kill-screen event, recover (problem_id, char_start, char_end)
# via hidden_reference's problem_id field (it IS carried there), then confirm
# against the label file's revisions list using marker_phrase + verified=true.
candidates = []
n_no_hidden, n_no_label_file, n_no_matching_revision, n_not_verified_correct, n_truncated = 0, 0, 0, 0, 0

for ev in kill_events:
    key = (ev["round"], ev["eid"])
    h = hidden.get(key)
    if h is None:
        n_no_hidden += 1
        continue
    model = ev["model"]
    pid = h["problem_id"]
    marker = h["marker_phrase"]

    label_obj = labels_index.get((model, pid))
    if label_obj is None:
        n_no_label_file += 1
        continue

    # find the specific revision: match marker_phrase, verified, and (as a
    # tie-breaker if multiple candidates share the same marker) prefer the one
    # whose confidence/reason matches hidden reference's recorded values
    revs = [r for r in label_obj.get("revisions", [])
            if r.get("matched_text") == marker and r.get("verified", True)]
    if not revs:
        n_no_matching_revision += 1
        continue
    had_multiple_candidates = len(revs) > 1
    exact_disambiguation_used = False
    if had_multiple_candidates:
        # disambiguate using confidence + reason from hidden reference, if available
        target_conf = h.get("confidence")
        target_reason = h.get("reason")
        exact = [r for r in revs if str(r.get("confidence")) == str(target_conf)
                 and r.get("reason") == target_reason]
        if exact:
            revs = exact
            exact_disambiguation_used = True
        else:
            revs = revs[:1]  # fall back to first candidate -- flagged below as unresolved

    rv = revs[0]
    trace_path = trace_index.get((model, pid))
    if trace_path is None:
        n_no_label_file += 1
        continue
    with open(trace_path) as fh:
        trace_obj = json.load(fh)

    if not trace_obj.get("correct", False):
        n_not_verified_correct += 1
        continue
    if trace_obj.get("truncated", False):
        n_truncated += 1
        continue

    candidates.append({
        "round": ev["round"], "eid": ev["eid"], "model": model, "problem_id": pid,
        "char_start": rv.get("char_start"), "char_end": rv.get("char_end"),
        "marker_phrase": marker, "human_label": ev["label"],
        "disambiguated_from_multiple": had_multiple_candidates,
        "disambiguation_unresolved": had_multiple_candidates and not exact_disambiguation_used,
        "problem": trace_obj.get("problem", ""),
        "answer_gt": trace_obj.get("answer_gt", ""),
        "difficulty": trace_obj.get("difficulty", "hard"),
        "full_response_len": len(trace_obj.get("full_response", "")),
    })

print(f"\nRecovery summary:")
print(f"  no hidden-reference match: {n_no_hidden}")
print(f"  no label file found: {n_no_label_file}")
print(f"  no matching verified revision: {n_no_matching_revision}")
print(f"  original trace not correct (excluded per design decision #3): {n_not_verified_correct}")
print(f"  original trace truncated (excluded, matches original pipeline's own convention): {n_truncated}")
print(f"  eligible candidates: {len(candidates)}")

# ---- Stratified pilot draw: balance local/global as much as available -----
random.seed(SEED)
locals_ = [c for c in candidates if c["human_label"] == "local"]
globals_ = [c for c in candidates if c["human_label"] == "global"]
print(f"\nEligible pool: {len(locals_)} local, {len(globals_)} global")

n_each = N_PILOT // 2
pilot_local = random.sample(locals_, min(n_each, len(locals_)))
pilot_global = random.sample(globals_, min(N_PILOT - len(pilot_local), len(globals_)))
pilot = pilot_local + pilot_global
random.shuffle(pilot)

print(f"Pilot set: {len(pilot)} events ({len(pilot_local)} local, {len(pilot_global)} global)")

# ---- Build prefix_response_text (the response text up to char_start) -----
# NOTE: trace files do NOT store the actual chat-templated prompt string --
# only prompt_tokens (a count). The real prompt must be reconstructed via
# build_prompt(problem, tokenizer, model_name) at generation time, using the
# "problem" and "difficulty" fields captured below -- do not use a stored
# "prompt" field, there isn't a real one to trust.
for c in pilot:
    trace_path = trace_index[(c["model"], c["problem_id"])]
    with open(trace_path) as fh:
        trace_obj = json.load(fh)
    full_response = trace_obj.get("full_response", "")
    cs = c["char_start"]
    c["prefix_response_text"] = full_response[:cs] if cs is not None else None

out_path = EXPORT_DIR / "counterfactual_pilot_events.json"
with open(out_path, "w") as f:
    json.dump(pilot, f, indent=2)
print(f"\nWrote pilot event set: {out_path}")

for c in pilot:
    if c["disambiguation_unresolved"]:
        flag = "  [MULTIPLE CANDIDATES, NO EXACT MATCH -- verify manually before using]"
    elif c["disambiguated_from_multiple"]:
        flag = "  [disambiguated among multiple candidates via confidence+reason match]"
    else:
        flag = ""
    print(f"  {c['model']:<28} {c['problem_id']:<14} label={c['human_label']:<7} "
          f"marker={c['marker_phrase']!r}{flag}")


## Step 2 — Generate suppressed rollouts (resumable)

**Safe to interrupt and re-run** -- it will skip whatever's already in `counterfactual_rollout_results.json` and only process what's left, including skipping model loads entirely for fully-completed models.

**Before running the full loop**: check the printed sanity check once the first model loads.

In [ ]:
# =============================================================================
# COUNTERFACTUAL NECESSITY -- suppressed-rollout generation
#
# Consumes counterfactual_pilot_events.json (from select_pilot_events.py).
# For each pilot event: reconstructs the prefix (prompt + response up to the
# revision), generates N rollouts with the revision-marker vocabulary banned,
# checks whether each rollout still reaches the correct answer, and writes
# counterfactual_rollout_results.json for validate_necessity.py to consume.
#
# build_prompt() and extract_answer() below are REUSED VERBATIM from
# Task_1_Data_Pipeline.ipynb (Cells 9 and 10) so "correct" means exactly the
# same thing here as it did when the original traces were generated -- not a
# reimplementation that could silently disagree on edge cases.
#
# CRITICAL DEVIATION FROM THE ORIGINAL PIPELINE, DELIBERATE AND DOCUMENTED:
# The original traces used greedy decoding (DO_SAMPLE=False, temperature=None)
# -- fine for a single deterministic trace, but useless here: N identical
# rollouts from the same truncated prefix with the same suppression would give
# a meaningless 0-or-1 necessity score instead of a real distribution. This
# script uses SAMPLING (temperature=0.6, top_p=0.95 -- DeepSeek's own general
# recommendation for their R1 models; adjust if you have a more specific
# figure) specifically for the counterfactual rollouts. This means rollout
# generation is NOT reproducing what "the model would have done" in a strict
# deterministic sense -- it's sampling from the model's distribution over
# continuations, which is what the necessity question actually needs.
#
# NOT YET TESTED END TO END (no GPU/HF access available where this was
# written) -- do a quick sanity check per the checklist at the bottom before
# committing to the full pilot run.
# =============================================================================

from pathlib import Path
import json, re, time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

GDRIVE_BASE = Path("/content/gdrive/MyDrive/llm_planning_study")
EXPORT_DIR = GDRIVE_BASE / "hand_labeling_export"
PILOT_EVENTS_PATH = EXPORT_DIR / "counterfactual_pilot_events.json"
RESULTS_PATH = EXPORT_DIR / "counterfactual_rollout_results.json"

N_ROLLOUTS = 8
ROLLOUT_MAX_NEW_TOKENS = 2048   # smaller than the original 8192 -- continuing
                                 # from partway through, not generating from scratch
SAMPLE_TEMPERATURE = 0.6
SAMPLE_TOP_P = 0.95

# ---- REUSED VERBATIM from Task_1_Data_Pipeline.ipynb, Cell 10 -------------
def extract_answer(text: str) -> str:
    boxed = re.findall(r'\\boxed\{([^}]+)\}', text)
    if boxed:
        return boxed[-1].strip()
    m = re.search(r'(?:answer is|therefore)[:\s]*([0-9A-Za-z\-\.\/]+)',
                  text, re.IGNORECASE)
    return m.group(1).strip() if m else "UNKNOWN"


# ---- REUSED VERBATIM from Task_1_Data_Pipeline.ipynb, Cell 9 --------------
def build_prompt(problem: dict, tokenizer, model_name: str) -> str:
    is_reasoning = any(tag in model_name for tag in ["R1", "QwQ"])

    if problem["difficulty"] == "hard":
        instruction = (
            "Solve the following competition mathematics problem carefully. "
            "Show your reasoning step by step and verify your answer."
        )
    else:
        instruction = (
            "Solve the following mathematics competition problem. "
            "Show your working step by step."
        )

    if is_reasoning:
        messages = [{"role": "user",
                     "content": f"{instruction}\n\nProblem: {problem['problem']}"}]
    else:
        messages = [
            {"role": "system",
             "content": "You are an expert mathematician. Solve problems with "
                        "clear step-by-step reasoning."},
            {"role": "user",
             "content": f"{instruction}\n\nProblem: {problem['problem']}"}
        ]

    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


# ---- Suppression vocabulary: the SAME regex used to detect candidate
# revision markers in the first place (Task_1_Data_Pipeline.ipynb, Cell 13 /
# REVISION_RE), so "suppressed" means the same thing as "detected" everywhere
# else in this project. Converted from a regex to an explicit phrase list for
# building bad_words_ids (tokenizer-level banning needs literal strings).
SUPPRESSED_PHRASES = [
    "wait", "Wait", "actually", "Actually", "let me reconsider",
    "that's wrong", "that is wrong", "i made an error", "I made an error",
    "let me try again", "this doesn't work", "this does not work",
    "i need to reconsider", "I need to reconsider", "hmm", "Hmm",
    "that's not right", "that is not right", "let me rethink",
    "on second thought", "let me go back", "wait, let me", "Wait, let me",
    "actually, let me", "Actually, let me", "let me recheck", "let me re-check",
    # Added after auditing suppression leaks -- these are confirmed real
    # matched_text values seen in this project's actual labels/*.json data
    # (e.g. during the type_hint vs verdict investigation) that REVISION_RE
    # from Task_1_Data_Pipeline.ipynb did NOT cover -- that regex is
    # apparently incomplete relative to whatever actually generated the real
    # marker detections. If more gaps turn up via audit_suppression_leaks.py,
    # add them here too rather than treating this list as final.
    "alternatively", "Alternatively", "I made a mistake", "i made a mistake",
    "I think I made a mistake", "let me double check", "let me double-check",
    "Let me double check", "Let me double-check",
    # Added after reading real rollout transcripts (Step 2.5) -- "hold on"
    # leaked repeatedly and was never on any version of this list.
    "hold on", "Hold on",
]


def build_bad_words_ids(tokenizer):
    """
    Builds token-id sequences to ban via generate()'s bad_words_ids. Tries each
    phrase with several boundary-context prefixes, since byte-level BPE
    tokenizers can merge preceding whitespace/punctuation into the SAME token
    as the following word -- e.g. "\\n\\nAlternatively" may be encoded as a
    compound token distinct from both "Alternatively" and " Alternatively".
    Missing a merge pattern means the ban silently never engages for that
    specific boundary -- this cannot be assumed complete, hence the
    verify-after-generation check in generate_suppressed_rollout below.
    """
    bad_words_ids = []
    seen = set()
    boundary_prefixes = ["", " ", "\n", "\n\n", ". ", ", ", ": "]
    for phrase in SUPPRESSED_PHRASES:
        for boundary in boundary_prefixes:
            variant = boundary + phrase
            ids = tuple(tokenizer.encode(variant, add_special_tokens=False))
            if ids and ids not in seen:
                seen.add(ids)
                bad_words_ids.append(list(ids))
    return bad_words_ids


def load_model_and_tokenizer(model_name: str):
    print(f"Loading tokenizer: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"Loading model (fp16)...")
    model = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.float16, device_map="auto",
        trust_remote_code=True,
    )
    model.eval()
    print("Model loaded.")
    return model, tokenizer


@torch.no_grad()
def generate_suppressed_rollout(model, tokenizer, prefix_text: str, bad_words_ids, seed: int):
    torch.manual_seed(seed)
    inputs = tokenizer(prefix_text, return_tensors="pt", truncation=False).to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=ROLLOUT_MAX_NEW_TOKENS,
        do_sample=True,
        temperature=SAMPLE_TEMPERATURE,
        top_p=SAMPLE_TOP_P,
        bad_words_ids=bad_words_ids,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    prompt_len = inputs["input_ids"].shape[1]
    new_tokens = output_ids[0][prompt_len:]
    full_response = tokenizer.decode(new_tokens, skip_special_tokens=False)
    return full_response


def main():
    assert PILOT_EVENTS_PATH.exists(), f"{PILOT_EVENTS_PATH} not found -- run select_pilot_events.py first"
    with open(PILOT_EVENTS_PATH) as f:
        pilot_events = json.load(f)
    print(f"Loaded {len(pilot_events)} pilot events.")

    by_model = {}
    for ev in pilot_events:
        by_model.setdefault(ev["model"], []).append(ev)

    # ---- resume support: skip events already completed in a previous run ----
    # Necessary at this scale (~140 events x up to 4 model loads) -- without
    # this, a Colab timeout partway through would silently redo everything,
    # including multi-GB model reloads for models already fully finished.
    all_results = []
    already_done = set()
    if RESULTS_PATH.exists():
        with open(RESULTS_PATH) as f:
            all_results = json.load(f)
        already_done = {(r["problem_id"], r["model"]) for r in all_results}
        print(f"Resuming from {RESULTS_PATH}: {len(already_done)} events already completed.")

    for model_name_short, events in by_model.items():
        remaining_events = [ev for ev in events if (ev["problem_id"], model_name_short) not in already_done]
        n_skipped = len(events) - len(remaining_events)
        if n_skipped:
            print(f"\n{model_name_short}: {n_skipped}/{len(events)} events already done, skipping those.")
        if not remaining_events:
            print(f"{model_name_short}: all events already done -- skipping model load entirely.")
            continue

        # events store the short name (e.g. "DeepSeek-R1-Distill-Qwen-1.5B");
        # the HF repo id needs the "deepseek-ai/" prefix to actually load
        model_name_full = f"deepseek-ai/{model_name_short}" if not model_name_short.startswith("deepseek-ai/") else model_name_short
        print(f"\n{'='*60}\n{model_name_full}  ({len(remaining_events)} events remaining)\n{'='*60}")

        model, tokenizer = load_model_and_tokenizer(model_name_full)
        bad_words_ids = build_bad_words_ids(tokenizer)
        print(f"Built {len(bad_words_ids)} banned token sequences.")
        print("Sanity check -- first 5 banned sequences, decoded back:")
        for ids in bad_words_ids[:5]:
            print(f"  {ids} -> {tokenizer.decode(ids)!r}")

        for ev in remaining_events:
            print(f"\n  {ev['problem_id']}  (human_label={ev['human_label']})")
            # Reconstruct the ACTUAL original prompt -- trace files never stored
            # the real chat-templated prompt text (only a token count), so this
            # must be rebuilt from problem+difficulty using the same function
            # that generated the original traces, not read from stored data.
            problem_dict = {"problem": ev["problem"], "difficulty": ev["difficulty"]}
            reconstructed_prompt = build_prompt(problem_dict, tokenizer, model_name_full)
            prefix_text = reconstructed_prompt + ev["prefix_response_text"]
            rollout_correct = []
            rollout_texts = []
            t0 = time.time()
            for r in range(N_ROLLOUTS):
                rollout_text = generate_suppressed_rollout(
                    model, tokenizer, prefix_text, bad_words_ids, seed=1000 + r
                )
                model_answer = extract_answer(rollout_text)
                is_correct = model_answer.strip() == ev["answer_gt"].strip()
                rollout_correct.append(is_correct)
                rollout_texts.append(rollout_text)
                print(f"    rollout {r+1}/{N_ROLLOUTS}: ans={model_answer!r} "
                      f"(gt={ev['answer_gt']!r}) correct={is_correct}")
            elapsed = time.time() - t0
            necessity = 1.0 - (sum(rollout_correct) / len(rollout_correct))
            print(f"  necessity={necessity:.3f}  ({elapsed:.1f}s for {N_ROLLOUTS} rollouts)")

            all_results.append({
                "problem_id": ev["problem_id"], "model": model_name_short,
                "human_label": ev["human_label"], "rollout_correct": rollout_correct,
                "rollout_texts": rollout_texts,
            })
            # checkpoint after every event, not just at the end
            with open(RESULTS_PATH, "w") as f:
                json.dump(all_results, f, indent=2)

        del model
        torch.cuda.empty_cache()

    print(f"\nWrote {len(all_results)} events' rollout results to {RESULTS_PATH}")


if __name__ == "__main__":
    main()


## Step 2.5 — Inspect specific rollouts (optional)

In [ ]:
# =============================================================================
# INSPECT SPECIFIC ROLLOUTS -- manual (a) vs (b) diagnostic
#
# The completed pilot run only saved rollout_correct (booleans), not the text
# itself -- generate_counterfactual_rollouts.py has been patched to save text
# going forward, but the already-completed run's transcripts are gone. This
# cell cheaply REGENERATES rollouts for just a handful of hand-picked events
# (the most and least "necessary" GLOBAL events, by the already-computed
# necessity scores) so you can read them directly.
#
# Reuses `model`, `tokenizer`, `bad_words_ids`, and the build_prompt /
# extract_answer / generate_suppressed_rollout functions already defined in
# the Step 2 cell above, IF that cell has been run in this session (avoids a
# redundant multi-GB model reload). Falls back to loading fresh if not.
#
# What to look for while reading: does the suppressed continuation actually
# proceed differently (new approach, no acknowledgment of a problem) -- or
# does it still "notice" something's off using unbanned words (e.g. "let me
# verify", "double-check", "that seems off", "alternatively")? The first
# pattern supports necessity as a real signal; the second suggests the ban
# list is leaky and activation steering is the next thing to try, not a
# verdict that necessity doesn't work.
# =============================================================================

import json
from pathlib import Path

GDRIVE_BASE = Path("/content/gdrive/MyDrive/llm_planning_study")
EXPORT_DIR = GDRIVE_BASE / "hand_labeling_export"
PILOT_EVENTS_PATH = EXPORT_DIR / "counterfactual_pilot_events.json"
RESULTS_PATH = EXPORT_DIR / "counterfactual_rollout_results.json"

N_INSPECT_PER_GROUP = 2   # how many highest- and lowest-necessity global events to read
N_ROLLOUTS_TO_REGEN = 3   # fresh rollouts per inspected event -- reading, not statistics

# words that suggest the model is still "noticing" a problem via unbanned
# vocabulary -- NOT used for suppression, only flagged here as a reading aid
LEAKAGE_FLAG_WORDS = [
    "verify", "double-check", "double check", "seems off", "seems wrong",
    "let me check", "re-examine", "reexamine", "start over", "scratch that",
    "different approach", "different method", "alternative approach", "instead, let's",
    "i realize", "i notice", "hold on", "not quite right", "doesn't seem",
    "let's redo", "recompute", "re-derive",
]


def flag_leakage(text):
    """Returns (word, context_snippet) for each hit -- includes surrounding text
    specifically so a flag pointing at content beyond the printed 600-char
    excerpt is still visible and interpretable, not just a bare word list."""
    hits = []
    text_lower = text.lower()
    for w in LEAKAGE_FLAG_WORDS:
        idx = text_lower.find(w.lower())
        if idx != -1:
            start = max(0, idx - 40)
            end = min(len(text), idx + len(w) + 40)
            snippet = text[start:end].replace("\n", " ")
            hits.append((w, f"...{snippet}..."))
    return hits


with open(RESULTS_PATH) as f:
    results = json.load(f)
with open(PILOT_EVENTS_PATH) as f:
    pilot_events = json.load(f)
pilot_by_id = {e["problem_id"]: e for e in pilot_events}

for r in results:
    r["necessity"] = 1.0 - (sum(r["rollout_correct"]) / len(r["rollout_correct"]))

import scope_flag
# Originally selected from GLOBAL-labelled events only, which made this
# diagnostic depend on the retired scope labels as a selection device. It now
# selects across ALL events by necessity extremes -- scope-free, and a better
# diagnostic: the leakage question this step answers has nothing to do with
# local/global.
if scope_flag.scope_banner():
    global_results = [r for r in results if r.get("human_label") == "global"]
    print(f"\nSelecting from GLOBAL-labelled events only "
          f"({len(global_results)} of {len(results)}) — retired selection.")
else:
    global_results = list(results)
    print(f"\nSelecting from ALL {len(global_results)} events by necessity "
          f"extremes (scope-free selection).")
global_results.sort(key=lambda r: r["necessity"])

lowest = global_results[:N_INSPECT_PER_GROUP]
highest = global_results[-N_INSPECT_PER_GROUP:]
to_inspect = lowest + highest
print("Selected for inspection:")
for r in to_inspect:
    tag = "LOW necessity" if r in lowest else "HIGH necessity"
    print(f"  {r['problem_id']}  necessity={r['necessity']:.3f}  ({tag})")

# ---- reuse model/tokenizer/bad_words_ids from Step 2 if already loaded -----
if "model" in globals() and "tokenizer" in globals() and "bad_words_ids" in globals():
    print("\nReusing already-loaded model/tokenizer/bad_words_ids from Step 2.")
else:
    print("\nStep 2's model isn't loaded in this session -- loading fresh "
          "(this will take a few minutes).")
    inspect_model_name = f"deepseek-ai/{to_inspect[0]['model']}"
    model, tokenizer = load_model_and_tokenizer(inspect_model_name)
    bad_words_ids = build_bad_words_ids(tokenizer)

# ---- regenerate and print ---------------------------------------------------
for r in to_inspect:
    ev = pilot_by_id[r["problem_id"]]
    tag = "LOW necessity" if r in lowest else "HIGH necessity"
    print(f"\n{'='*70}\n{ev['problem_id']}  necessity={r['necessity']:.3f}  ({tag})\n{'='*70}")
    print(f"Problem: {ev['problem'][:200]}")
    print(f"Answer (ground truth): {ev['answer_gt']}")
    print(f"\n--- End of prefix (last 300 chars before the suppressed revision) ---")
    print(ev["prefix_response_text"][-300:])

    problem_dict = {"problem": ev["problem"], "difficulty": ev["difficulty"]}
    inspect_model_name = f"deepseek-ai/{ev['model']}"
    reconstructed_prompt = build_prompt(problem_dict, tokenizer, inspect_model_name)
    prefix_text = reconstructed_prompt + ev["prefix_response_text"]

    for i in range(N_ROLLOUTS_TO_REGEN):
        rollout_text = generate_suppressed_rollout(
            model, tokenizer, prefix_text, bad_words_ids, seed=5000 + i
        )
        model_answer = extract_answer(rollout_text)
        is_correct = model_answer.strip() == ev["answer_gt"].strip()
        hits = flag_leakage(rollout_text)
        print(f"\n  --- rollout {i+1}/{N_ROLLOUTS_TO_REGEN} "
              f"(ans={model_answer!r}, correct={is_correct}) ---")
        print(f"  {rollout_text[:600]}")
        if hits:
            print(f"  >>> POSSIBLE LEAKAGE -- {len(hits)} unbanned hedge word(s) found:")
            for word, snippet in hits:
                print(f"      {word!r}: {snippet}")
        else:
            print(f"  (no flagged hedge words found)")


## Step 2.6 — Audit suppression leaks (recommended before Step 3)

In [ ]:
# =============================================================================
# SUPPRESSION LEAK AUDIT
#
# Scans ALL saved rollout text (requires having re-run generation with the
# patched script that saves rollout_texts) for occurrences of:
#   (a) EXPLICITLY BANNED phrases -- these should never appear. If they do,
#       that's a suppression-mechanism failure, not the anticipated leakage.
#   (b) KNOWN-BUT-UNBANNED phrases (e.g. "alternatively", which real labeled
#       data confirms is a genuine marker, but was missing from the ban list)
#       -- these appearing is the anticipated (b) leakage from the design doc,
#       not a bug, but still informative.
#
# Turns "I think I still see revision words" into an exact count: which
# phrase, how many rollouts, which events -- before deciding whether this is
# a fixable ban-list gap or a deeper mechanism problem worth escalating to
# activation steering.
# =============================================================================

import json
import re
from pathlib import Path
from collections import Counter

GDRIVE_BASE = Path("/content/gdrive/MyDrive/llm_planning_study")
EXPORT_DIR = GDRIVE_BASE / "hand_labeling_export"
RESULTS_PATH = EXPORT_DIR / "counterfactual_rollout_results.json"

# Must match SUPPRESSED_PHRASES in generate_counterfactual_rollouts.py exactly --
# update BOTH lists together if you change one.
EXPLICITLY_BANNED = [
    "wait", "actually", "let me reconsider", "that's wrong", "that is wrong",
    "i made an error", "let me try again", "this doesn't work",
    "this does not work", "i need to reconsider", "hmm", "that's not right",
    "that is not right", "let me rethink", "on second thought",
    "let me go back", "wait, let me", "actually, let me", "let me recheck",
    "let me re-check", "alternatively", "i made a mistake",
    "i think i made a mistake", "let me double check", "let me double-check",
    "hold on",
]

# Phrases NOT currently banned -- populate this as new gaps are found via
# auditing (e.g. by reading rollouts in Step 2.5). Empty for now since the
# known gaps found so far have been added to the ban list above instead.
KNOWN_UNBANNED_MARKERS = []


def scan(text, phrase_list):
    text_lower = text.lower()
    return [p for p in phrase_list if p.lower() in text_lower]


def main():
    assert RESULTS_PATH.exists(), f"{RESULTS_PATH} not found"
    with open(RESULTS_PATH) as f:
        results = json.load(f)

    events_with_text = [r for r in results if r.get("rollout_texts")]
    if not events_with_text:
        print("No rollout_texts found in results -- this file was written before the text-saving "
              "patch, or generation hasn't been re-run yet. Re-run Step 2 to get scannable text.")
        return

    print(f"Scanning {len(events_with_text)} events with saved rollout text "
          f"({sum(len(r['rollout_texts']) for r in events_with_text)} total rollouts)...")

    banned_hits = Counter()
    unbanned_hits = Counter()
    banned_hit_events = set()
    unbanned_hit_events = set()
    n_rollouts_with_any_banned_leak = 0

    for r in events_with_text:
        for rollout_text in r["rollout_texts"]:
            banned_found = scan(rollout_text, EXPLICITLY_BANNED)
            unbanned_found = scan(rollout_text, KNOWN_UNBANNED_MARKERS)
            for p in banned_found:
                banned_hits[p] += 1
                banned_hit_events.add(r["problem_id"])
            for p in unbanned_found:
                unbanned_hits[p] += 1
                unbanned_hit_events.add(r["problem_id"])
            if banned_found:
                n_rollouts_with_any_banned_leak += 1

    total_rollouts = sum(len(r["rollout_texts"]) for r in events_with_text)

    print(f"\n{'='*70}\nEXPLICITLY BANNED PHRASES -- should be ZERO. Any hit is a mechanism failure.\n{'='*70}")
    if banned_hits:
        for phrase, count in banned_hits.most_common():
            print(f"  {phrase!r}: appeared in {count} rollouts")
        print(f"\n  {n_rollouts_with_any_banned_leak}/{total_rollouts} rollouts "
              f"({n_rollouts_with_any_banned_leak/total_rollouts:.0%}) contained at least one "
              f"explicitly-banned phrase.")
        print(f"  Affected events: {sorted(banned_hit_events)}")
        print("\n  >>> This means bad_words_ids did not reliably suppress its own ban list.")
        print("  >>> Likely causes: tokenization variants not covered (e.g. banned phrase merged")
        print("      with adjacent punctuation into a single token not in the ban list), or a")
        print("      library/version issue with how bad_words_ids is enforced. Worth checking the")
        print("      Step 2 sanity printout (decoded banned sequences) actually looked right, and")
        print("      considering a custom LogitsProcessor that checks generated text post-hoc at")
        print("      each step rather than relying on exact pre-tokenized sequence matching.")
    else:
        print("  None found. Suppression mechanism worked on its own ban list.")

    print(f"\n{'='*70}\nKNOWN-BUT-UNBANNED MARKERS -- expected leakage, not a bug, but informative\n{'='*70}")
    if unbanned_hits:
        for phrase, count in unbanned_hits.most_common():
            print(f"  {phrase!r}: appeared in {count} rollouts")
        print(f"  Affected events: {sorted(unbanned_hit_events)}")
    else:
        print("  None found.")

    print(f"\n{'#'*70}\n# READ\n{'#'*70}")
    if banned_hits:
        print("Explicitly-banned phrases leaking through is the more serious finding -- this points")
        print("toward (b) suppression-mechanism failure specifically, not just (a) necessity being a")
        print("weak proxy. Fix the mechanism (or escalate to activation steering) before drawing any")
        print("conclusion about necessity itself from this pilot's AUC.")
    elif unbanned_hits:
        print("Only known-unbanned markers leaked -- consistent with the design doc's anticipated")
        print("(b) leakage. Add these to SUPPRESSED_PHRASES in generate_counterfactual_rollouts.py")
        print("and re-run before drawing conclusions.")
    else:
        print("No leakage detected in either category. The AUC result can be read at face value.")


if __name__ == "__main__":
    main()


## Step 3 — Validate necessity against human labels

Filters out individually-leaked rollouts before computing necessity.

## ⚠️ Retired — necessity vs human scope labels

This step scores necessity against the human **local/global** labels. That analysis is retired: all three scope operationalizations returned nulls and the paper makes no scope claim.

It is **skipped by default** and left here for the record. The live validity control for this mechanism is the matched-filler comparison, in `Paper2_3_Lexical_Filler_Control.ipynb` (lexical) and `Paper2_4_Steering_Powered_Run.ipynb` (steering).

To run it anyway, set `SCOPE_ANALYSIS_ENABLED = True` in the setup cell.


In [ ]:
import scope_flag
# RETIRED — this step measures necessity against the human local/global SCOPE
# labels. The paper makes no scope claim, so it is skipped by default. The LIVE
# validity control for this mechanism is the matched-filler comparison:
#   lexical  -> Paper2_3_Lexical_Filler_Control.ipynb
#   steering -> Paper2_4_Steering_Powered_Run.ipynb
# Body left byte-identical below (it contains triple-quoted strings, so it is
# guarded here rather than indented).
if not scope_flag.scope_banner():
    scope_flag.skip_note("necessity vs human local/global labels")
    raise SystemExit("Retired scope analysis — skipped by design, not an error.")

# =============================================================================
# COUNTERFACTUAL NECESSITY -- validation step
#
# Consumes rollout results (produced by YOUR suppressed-generation code, which
# depends on your inference setup and isn't included here -- see the design
# doc's "open dependency" section). Expected input format, one JSON list:
#
#   [
#     {"problem_id": "...", "model": "...", "human_label": "local"|"global",
#      "rollout_correct": [true, false, true, ...]},   # one bool per rollout
#     ...
#   ]
#
# Computes necessity = 1 - fraction(rollout_correct == True), then checks
# whether necessity separates human-labeled local vs global -- same AUC /
# Mann-Whitney machinery as the embedding-divergence kill screen, for direct
# comparability between the two approaches.
# =============================================================================

import json
from pathlib import Path
import numpy as np
from sklearn.metrics import roc_auc_score
from scipy.stats import mannwhitneyu

GDRIVE_BASE = Path("/content/gdrive/MyDrive/llm_planning_study")
EXPORT_DIR = GDRIVE_BASE / "hand_labeling_export"
ROLLOUT_RESULTS_PATH = EXPORT_DIR / "counterfactual_rollout_results.json"

# Same list as audit_suppression_leaks.py's EXPLICITLY_BANNED -- keep in sync.
EXPLICITLY_BANNED = [
    "wait", "actually", "let me reconsider", "that's wrong", "that is wrong",
    "i made an error", "let me try again", "this doesn't work",
    "this does not work", "i need to reconsider", "hmm", "that's not right",
    "that is not right", "let me rethink", "on second thought",
    "let me go back", "wait, let me", "actually, let me", "let me recheck",
    "let me re-check", "alternatively", "i made a mistake",
    "i think i made a mistake", "let me double check", "let me double-check",
    "hold on",
]


def is_leaked(rollout_text):
    text_lower = rollout_text.lower()
    return any(p.lower() in text_lower for p in EXPLICITLY_BANNED)


def filter_leaked_rollouts(results):
    """Drops individual leaked rollouts (not whole events) before computing
    necessity. Events that lose ALL rollouts to leaks are excluded entirely
    and reported separately -- their necessity would be undefined, not zero."""
    filtered_results = []
    n_rollouts_dropped = 0
    n_events_fully_dropped = 0

    for r in results:
        if "rollout_texts" not in r:
            # no text saved (pre-patch run) -- can't filter, pass through as-is
            filtered_results.append(r)
            continue
        clean_correct = []
        for correct, text in zip(r["rollout_correct"], r["rollout_texts"]):
            if is_leaked(text):
                n_rollouts_dropped += 1
            else:
                clean_correct.append(correct)
        if not clean_correct:
            n_events_fully_dropped += 1
            print(f"  WARNING: {r['problem_id']} lost ALL rollouts to leakage -- excluded entirely")
            continue
        new_r = dict(r)
        new_r["rollout_correct"] = clean_correct
        new_r["n_dropped"] = len(r["rollout_correct"]) - len(clean_correct)
        filtered_results.append(new_r)

    print(f"Filtering: dropped {n_rollouts_dropped} individually-leaked rollouts, "
          f"{n_events_fully_dropped} events fully excluded (lost all rollouts).")
    return filtered_results


def compute_necessity(rollout_correct):
    if len(rollout_correct) == 0:
        return None
    return 1.0 - (sum(rollout_correct) / len(rollout_correct))


def report(results, tier_name):
    print(f"\n{'='*70}\n{tier_name}  (n={len(results)})\n{'='*70}")
    local_nec = [r["necessity"] for r in results if r["human_label"] == "local"]
    global_nec = [r["necessity"] for r in results if r["human_label"] == "global"]
    print(f"  n_local={len(local_nec)}  n_global={len(global_nec)}")
    if len(local_nec) < 2 or len(global_nec) < 2:
        print("  TOO FEW EVENTS IN ONE CLASS -- cannot compute AUC reliably.")
        return None
    print(f"  median necessity -- local: {np.median(local_nec):.3f}   global: {np.median(global_nec):.3f}")
    labels = [1]*len(global_nec) + [0]*len(local_nec)
    scores = global_nec + local_nec
    auc = roc_auc_score(labels, scores)
    u_stat, p_val = mannwhitneyu(global_nec, local_nec, alternative='two-sided')
    print(f"  AUC (necessity predicting global): {auc:.3f}")
    print(f"  Mann-Whitney U p-value: {p_val:.4f}")
    return auc


def main():
    with open(ROLLOUT_RESULTS_PATH) as f:
        results = json.load(f)
    print(f"Loaded {len(results)} events with rollout results.")

    results = filter_leaked_rollouts(results)
    print(f"{len(results)} events remain after filtering.\n")

    for r in results:
        r["necessity"] = compute_necessity(r["rollout_correct"])
        r["n_rollouts"] = len(r["rollout_correct"])

    n_rollouts_seen = set(r["n_rollouts"] for r in results)
    if len(n_rollouts_seen) > 1:
        print(f"NOTE: rollout counts vary across events: {sorted(n_rollouts_seen)}. "
              f"Necessity is a fraction so this is comparable across events, but low "
              f"rollout counts (e.g. <5) make individual necessity scores noisy -- read "
              f"any single event's score with that in mind.")

    auc = report(results, "COUNTERFACTUAL NECESSITY PILOT")

    print(f"\n{'#'*70}\n#  DECISION (per pre-committed pilot design)\n{'#'*70}")
    if auc is None:
        print("Could not compute AUC (insufficient class balance). Check pilot event selection.")
    elif auc >= 0.75:
        print(f"AUC={auc:.3f} -- comfortably separates. PROMISING. Scale to the full 200-event "
              f"set with more rollouts per event before concluding anything stronger.")
    else:
        print(f"AUC={auc:.3f} -- near chance / weak. Two live explanations, not one:")
        print("  (a) necessity genuinely isn't a good scope proxy, or")
        print("  (b) token-banning suppression was too leaky to test it properly.")
        print("  Before shelving: manually read a few suppressed rollouts for events with high")
        print("  necessity -- do they actually look like they avoided the revision's effect, or")
        print("  does the model 'notice' the same issue in different words? That distinguishes")
        print("  (a) from (b) and determines whether activation steering is worth trying next.")


if __name__ == "__main__":
    main()
